In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Suppressing warnings for cleaner output
import warnings
warnings.filterwarnings('ignore')

Matplotlib is building the font cache; this may take a moment.


In [3]:
# Reading only the first 1000 rows to understand the structure
df_sample = pd.read_csv('../data/rba-dataset.csv', nrows=1000)

print("Shape:", df_sample.shape)
print("\nColumns:", df_sample.columns.tolist())
print("\nData types:\n", df_sample.dtypes)
print("\nFirst 51 rows:")
df_sample.head(51)

Shape: (1000, 16)

Columns: ['index', 'Login Timestamp', 'User ID', 'Round-Trip Time [ms]', 'IP Address', 'Country', 'Region', 'City', 'ASN', 'User Agent String', 'Browser Name and Version', 'OS Name and Version', 'Device Type', 'Login Successful', 'Is Attack IP', 'Is Account Takeover']

Data types:
 index                         int64
Login Timestamp                 str
User ID                       int64
Round-Trip Time [ms]        float64
IP Address                      str
Country                         str
Region                          str
City                            str
ASN                           int64
User Agent String               str
Browser Name and Version        str
OS Name and Version             str
Device Type                     str
Login Successful               bool
Is Attack IP                   bool
Is Account Takeover            bool
dtype: object

First 51 rows:


,index,Login Timestamp,User ID,Round-Trip Time [ms],IP Address,Country,Region,City,ASN,User Agent String,Browser Name and Version,OS Name and Version,Device Type,Login Successful,Is Attack IP,Is Account Takeover
0,0,2020-02-03 12:43:30.772,-4324475583306591935,NaN,10.0.65.171,NO,-,-,29695,Mozilla/5.0 (iPhone; CPU iPhone OS 13_4 like ...,Firefox 20.0.0.1618,iOS 13.4,mobile,False,False,False
1,1,2020-02-03 12:43:43.549,-4324475583306591935,NaN,194.87.207.6,AU,-,-,60117,Mozilla/5.0 (Linux; Android 4.1; Galaxy Nexus...,Chrome Mobile 46.0.2490,Android 4.1,mobile,False,False,False
2,2,2020-02-03 12:43:55.873,-3284137479262433373,NaN,81.167.144.58,NO,Vestland,Urangsvag,29695,Mozilla/5.0 (iPad; CPU OS 7_1 like Mac OS X) ...,Android 2.3.3.2672,iOS 7.1,mobile,True,False,False
3,3,2020-02-03 12:43:56.180,-4324475583306591935,NaN,170.39.78.152,US,-,-,393398,Mozilla/5.0 (Linux; Android 4.1; Galaxy Nexus...,Chrome Mobile WebView 85.0.4183,Android 4.1,mobile,False,False,False
4,4,2020-02-03 12:43:59.396,-4618854071942621186,NaN,10.0.0.47,US,Virginia,Ashburn,398986,Mozilla/5.0 (Linux; U; Android 2.2) Build/NMA...,Chrome Mobile WebView 85.0.4183,Android 2.2,mobile,False,True,False
5,5,2020-02-03 12:44:05.160,-4324475583306591935,NaN,209.236.123.126,US,-,-,393398,Mozilla/5.0 (Linux; Android 4.1; Galaxy Nexus...,Chrome Mobile WebView 85.0.4183,Android 4.1,mobile,False,True,False
6,6,2020-02-03 12:44:07.893,7246533443898239661,NaN,80.202.228.214,NO,-,-,15659,Mozilla/5.066704189 Mozilla/5.0 (X11; CrOS x8...,Chrome 69.0.3497.17.19,Chrome OS 13020.67.3,desktop,True,False,False
7,7,2020-02-03 12:44:12.799,-3243978724802435038,NaN,170.39.78.177,US,-,-,393398,Mozilla/5.0 (Linux; Android 10.0.99; H30-U10) ...,Chrome Mobile 81.0.4044.1931,Android 10.0.99,mobile,False,False,False
8,8,2020-02-03 12:44:17.474,8076000552587369902,NaN,10.0.61.212,NO,-,-,29695,Mozilla/5.0 (Linux; U; Android 7.1.2; en-gb; R...,MiuiBrowser 12.6.5,Android 7.1.2,mobile,False,False,False
9,9,2020-02-03 12:44:19.071,-3065936140549856249,NaN,92.221.109.162,NO,Rogaland,Sandnes,29695,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_14_6...,Chrome 69.0.3497.17.28,Mac OS X 10.14.6,desktop,True,False,False


In [4]:
print("Missing values per column:")
print(df_sample.isnull().sum())
print("\nMissing percentage:")
print((df_sample.isnull().sum() / len(df_sample) * 100).round(2))

Missing values per column:
index                         0
Login Timestamp               0
User ID                       0
Round-Trip Time [ms]        965
IP Address                    0
Country                       0
Region                        1
City                          0
ASN                           0
User Agent String             0
Browser Name and Version      0
OS Name and Version           0
Device Type                   0
Login Successful              0
Is Attack IP                  0
Is Account Takeover           0
dtype: int64

Missing percentage:
index                        0.0
Login Timestamp              0.0
User ID                      0.0
Round-Trip Time [ms]        96.5
IP Address                   0.0
Country                      0.0
Region                       0.1
City                         0.0
ASN                          0.0
User Agent String            0.0
Browser Name and Version     0.0
OS Name and Version          0.0
Device Type                  0.

In [5]:
# Count total rows without loading everything
chunk_size = 100_000
total_rows = 0
attack_rows = 0
successful_rows = 0
countries = set()

for chunk in pd.read_csv('../data/rba-dataset.csv', chunksize=chunk_size):
    total_rows += len(chunk)
    attack_rows += chunk['Is Attack IP'].sum()
    successful_rows += chunk['Login Successful'].sum()
    countries.update(chunk['Country'].dropna().unique())

print(f"Total rows: {total_rows:,}")
print(f"Attack IPs: {attack_rows:,} ({attack_rows/total_rows*100:.2f}%)")
print(f"Successful logins: {successful_rows:,} ({successful_rows/total_rows*100:.2f}%)")
print(f"Unique countries: {len(countries)}")

Total rows: 31,269,264
Attack IPs: 3,096,977 (9.90%)
Successful logins: 12,541,442 (40.11%)
Unique countries: 229


In [6]:
# Sampling 500,000 rows randomly
# Set random_state for reproducibility
sampled_chunks = []
chunk_size = 100_000
sample_fraction = 0.05  # 5% of each chunk

for chunk in pd.read_csv('../data/rba-dataset.csv', chunksize=chunk_size):
    sampled_chunks.append(chunk.sample(frac=sample_fraction, random_state=42))

df = pd.concat(sampled_chunks, ignore_index=True)
print(f"Working dataset size: {df.shape}")
print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.1f} MB")

# Save the sample so you don't have to re-read the 9GB file every time
df.to_csv('../data/rba_sample.csv', index=False)
print("Sample saved to data/rba_sample.csv")

Working dataset size: (1563463, 16)
Memory usage: 1045.8 MB
Sample saved to data/rba_sample.csv
